In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
# from models.onedcnn_model import CNN_ForecastNet
from ptflops import get_model_complexity_info

In [2]:
class SFGRU(nn.Module):
    """
    An encoder-decoder model for pedestrian trajectory prediction.

    Attributes:
        _num_hidden_units: Number of GRU hidden units.
        _regularizer_value: The value of L2 regularizer for training.
        _regularizer: Training regularizer set as L2.

    Methods:
        stacked_rnn: Generates the network model.
        _gru: A helper function for creating a GRU unit.
    """

    def __init__(self, x1_data_sizes, x2_data_size, num_hidden_units=4):
        super(SFGRU, self).__init__()
        # Network parameters
        self._num_hidden_units = num_hidden_units
        self.grus = nn.ModuleList()  # To store the GRU layers
        self.dense = nn.Linear(num_hidden_units+x2_data_size[-1], 1)
        self.num_layers = len(x1_data_sizes)
        
        for i in range(self.num_layers):
            if i == 0:
                first_data = x1_data_sizes[i][1]
                self.grus.append(nn.GRU(input_size=first_data, hidden_size=num_hidden_units, batch_first=True))
            else:
                self.grus.append(nn.GRU(input_size=num_hidden_units + x1_data_sizes[i][1], hidden_size=num_hidden_units, batch_first=True))
        
        

    def forward(self, x):
        """
        x should be a list of data
        """
        x1 = x[0]
        x2 = x[1]
#         print(x1[0].shape, x2.shape)
        num_layers = self.num_layers
        for i in range(self.num_layers):            
            if i == 0:
                out, _ = self.grus[i](input=x1[i])
            elif i<num_layers-1:
                cat = torch.cat((x1[i], out), dim=2)
                out, _ = self.grus[i](input=cat)
            else:#last layer
                cat = torch.cat((x1[i], out), dim=2)
                _, hn = self.grus[i](input=cat)                
        
        hn = torch.squeeze(hn, dim=1)
#         print(hn.shape, x2.shape)
        cat = torch.cat([hn,x2], dim=1)        
        model_output = self.dense(cat)  # Dense layer applied to the last output in the sequence
        return model_output

In [3]:
def prepare_input(resolution):
    x1 = [torch.rand(resolution), torch.rand(resolution)]
    x2 = torch.rand(1, 2)
    return dict(x = [x1, x2])

In [4]:
x1_data_sizes = [(32, 1), (32, 1)]
x2_data_size =  torch.rand(1, 2).shape

model = SFGRU(x1_data_sizes, x2_data_size)
macs, params = get_model_complexity_info(model, input_res=(1, 32, 1), 
                                          input_constructor=prepare_input,
                                          as_strings=True, print_per_layer_stat=False)
print('      - macs:  ' + macs)
print('      - Params: ' + params)

Flops estimation was not finished successfully because of the following exception: 
<class 'IndexError'>: tuple index out of range


Traceback (most recent call last):
  File "/data/home/acw607/stacking/lib/python3.10/site-packages/ptflops/pytorch_engine.py", line 66, in get_flops_pytorch
    _ = flops_model(**batch)
  File "/data/home/acw607/stacking/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1736, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/data/home/acw607/stacking/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1844, in _call_impl
    return inner()
  File "/data/home/acw607/stacking/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1790, in inner
    result = forward_call(*args, **kwargs)
  File "/tmp/ipykernel_3157179/3966643598.py", line 42, in forward
    out, _ = self.grus[i](input=x1[i])
  File "/data/home/acw607/stacking/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1736, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/data/home/acw607/stacking/lib/python3.10/site-packages/torch/nn/mo

TypeError: can only concatenate str (not "NoneType") to str